# 09C 演示完整学习，迁移到另一个仓库

完整演示要有前后状态：资料导读、练习要求、错误尝试、一级提示、学生修改、再次运行和继续学习。只换一个仓库名称而没有重新核查结果，不算完成迁移。

两组示例都是课程随附、可在本地复核的最小案例。第八节已经真实导读 Mini Guide、Flashcards 和固定版本 Hello-Agents；这里把练习反馈迁移到第二个案例仓库，并明确哪些流程可以复用、哪些契约必须重建。

**最终验收**：第八节交付“有证据的导读基线”，第九节在其上增加“练习—反馈—状态—迁移”闭环。最终结果必须同时保留失败尝试、修订后通过、跨进程继续和有区分力的回归用例。

**贯穿案例**

把两道真实执行过的练习串成“提交—测试—按失败给提示—修改—重测—保存状态”的闭环。CLI 在另一个进程继续同一学习记录；回归用例专门检查今天到期的卡是否保留，也说明单个用例拦不住哪些别的错误。迁移报告写清哪些环节已执行，哪些只是迁移设计。


## 学习路线与配套课件

本 Notebook 可以独立从头运行。先读定义和输入，预测结果，再执行并解释差异。小算例帮助理解机制；真实训练、工具执行与保存的 API 记录会分别标明来源。

本节重点：到期当天，怎样拦住 < 与 <= 的错误。

课件页：L09.07-S01、L09.07-S02、L09.08-S02。

沿原有课程单元顺序完成实验；新增对照放在相关代码后。练习先自行作答，展开参考分析后再比较。

In [1]:
from pathlib import Path
import sys, json, tempfile, subprocess, inspect, hashlib
from datetime import datetime, timezone
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
PROJECT = ROOT / "09 综合实践二 练习反馈与跨仓库迁移/代码/学习反馈"
SOURCE_PROJECT = ROOT / "08 综合实践一 资料与仓库导读/代码/学习助手"
sys.path.insert(0,str(SOURCE_PROJECT/"src"))
sys.path.insert(0,str(PROJECT/"src"))
from exercise_lab import LearningStore,run_exercise,catalog,CATALOG,verified_feedback
from corpus import Corpus
from tutor_runtime import TutorRuntime
from IPython.display import display,Code
print("默认离线；只执行课程自编的小练习，不执行资料仓库中的任意代码。")


默认离线；只执行课程自编的小练习，不执行资料仓库中的任意代码。


## 1. 按证据讲一轮完整学习

先看第八节导读，再读本节两次反馈。不要求模型每次使用相同工具序列，而是核查：读了当前尝试、提示有帮助、修改后只宣称用例通过、保留了版本与失败。

下面统计的是四次反馈任务里的模型请求，不把每个反馈任务都当成只调用一次 API。


In [2]:
report=json.loads((PROJECT/"results/deepseek-feedback.json").read_text(encoding='utf-8'))
for case in report["cases"]:
    print("\n练习:",case["exercise_id"])
    for phase in case["phases"]:
        state=phase["state"];learning=phase["learning"]
        print(phase["phase"],state["status"],"请求",state["turns"],"工具",state["tool_calls"])
        print("保留尝试:",len(learning["attempts"]),"当前学习状态:",learning["status"])
        print(state["answer"])
        assert learning["lesson_id"]==case["lesson_id"]
focused=json.loads((PROJECT/"results/deepseek-feedback-focused.json").read_text(encoding='utf-8'))
print("日期练习修订后的状态:",[p["state"]["status"] for p in focused["cases"][0]["phases"]])
print("任务完成仍有教学错误：实际 2 项失败被说成 4 项，请依据保存的程序证据自行逐条复核。")



练习: rank-positive
needs_work final_answer 请求 3 工具 2
保留尝试: 1 当前学习状态: needs_work
当前测试：稳定同分、limit=1、空列表已通过；过滤零分、降序、非法 limit、布尔 limit 未通过。

提示：零分项出现在结果里，说明过滤条件写错了方向——score<=0 应被剔除。另外检查 limit 判断：True 是布尔，需显式排除 bool 类型，再做正整数校验和截断。

试试自己改，改完再跑一次。
tests_passed final_answer 请求 2 工具 1
保留尝试: 2 当前学习状态: tests_passed
已读取最新记录：随课用例全部通过（过滤非正分、降序、同分稳定、limit 截断、空列表，以及非正整数/布尔 limit 抛 ValueError）。这只说明本轮随课测试通过，不代表已完全掌握。

自检：limit 传入 True 时你走的是哪条分支？为什么普通 `isinstance(limit, int)` 会误放行布尔值？

练习: due-cards
needs_work final_answer 请求 2 工具 2
保留尝试: 1 当前学习状态: needs_work
本轮测试未通过，`equal-today` 和 `stable-tie` 两个用例返回了空列表。

提示：先检查筛选条件——`due` 恰好等于 `today` 的那张卡，按题意应不应该被选中？用 `<=` 还是 `<`，会直接决定这两组用例的结果。再看 `stable-tie` 里同一天的卡片是否因为排序方式而丢掉了顺序。

请自己核对条件与排序稳定性后再试一次。
tests_passed final_answer 请求 2 工具 1
保留尝试: 2 当前学习状态: tests_passed
本轮随课测试全部通过：等于今天的计入、未来卡排除、按 due 升序、limit 截断、同日保持原顺序、非法 limit 抛 ValueError。

这只能说明随课用例通过，不等于已掌握。请自检：同一天多张卡时，你的实现靠什么保证原顺序？如果改用 set 或先排序再按 due 分组，这个性质还成立吗？
日期练习修订后的状态: ['final_answer', 'final_answer']
任务完成仍有

## 2. 换的是资料与题目，复用的是流程

Mini Guide 的知识点是过滤、稳定排序和 limit；Flashcards 的知识点是日期边界、排序和 limit。同一个 Corpus、Harness、学习记录与执行器接收不同配置。

复用程序不等于无需核查题目。日期比较按题目前提使用有效 YYYY-MM-DD 字符串；不推广成任意字符串都能正确比较日期。


In [3]:
rows=[]
for exercise_id,exercise in catalog().items():
    for file in ["student.py","teaching_revision.py"]:
        result=run_exercise(exercise_id,PROJECT/"exercises"/exercise_id/file)
        tests=result["result"]["tests"]
        row={"exercise":exercise_id,"submission":file,"passed":sum(t["passed"] for t in tests),"total":len(tests)}
        rows.append(row);print(row)
        assert result["result"]["passed"]==(file=="teaching_revision.py")
print("这是两类小练习的回归；不是整个通用辅导系统质量的充分证明。")


{'exercise': 'rank-positive', 'submission': 'student.py', 'passed': 3, 'total': 7}
{'exercise': 'rank-positive', 'submission': 'teaching_revision.py', 'passed': 7, 'total': 7}
{'exercise': 'due-cards', 'submission': 'student.py', 'passed': 4, 'total': 6}


{'exercise': 'due-cards', 'submission': 'teaching_revision.py', 'passed': 6, 'total': 6}
这是两类小练习的回归；不是整个通用辅导系统质量的充分证明。


## 3. 修改条件，要让输出能看出差异

如果本来只有两张到期卡，把 limit 从 3 改成 2，输出可能不变。这不能说明限制不起作用。第八节真实导读给出的练习就需要改进。

下面把 limit 改成 1，再分别改变 today，观察限制与筛选的不同作用。


In [4]:
import importlib.util
path=PROJECT/"exercises/due-cards/teaching_revision.py"
spec=importlib.util.spec_from_file_location("trusted_due_solution",path)
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
cards=[{"id":"yesterday","due":"2026-09-06"},{"id":"today","due":"2026-09-07"},{"id":"future","due":"2026-09-09"}]
for today,limit in [("2026-09-07",3),("2026-09-07",2),("2026-09-07",1),("2026-09-06",3)]:
    print(today,limit,[c["id"] for c in module.solve(cards,today,limit)])
assert len(module.solve(cards,"2026-09-07",1))==1


2026-09-07 3 ['yesterday', 'today']
2026-09-07 2 ['yesterday', 'today']
2026-09-07 1 ['yesterday']
2026-09-06 3 ['yesterday']


### 到期当天，怎样拦住 < 与 <= 的错误

配套课件：L09.07-S01、L09.07-S02、L09.08-S02。

在迁移后的日期题里，due 等于 today 的卡也到期。将过滤条件故意改成严格小于，构造一对可比较输出。输入日期采用 YYYY-MM-DD，按课程约定可用字符串顺序比较。

In [5]:
case_today = "2026-09-07"
case_cards = [{"id":"yesterday", "due":"2026-09-06"},
              {"id":"today", "due":case_today}, {"id":"future", "due":"2026-09-08"}]
case_correct = module.solve(case_cards, case_today, 3)
case_mutant = sorted((c for c in case_cards if c["due"] < case_today), key=lambda c:c["due"])[:3]
print("正确:", [c["id"] for c in case_correct])
print("错误 < 条件:", [c["id"] for c in case_mutant])
assert [c["id"] for c in case_correct] == ["yesterday", "today"]
assert [c["id"] for c in case_mutant] == ["yesterday"]


正确: ['yesterday', 'today']
错误 < 条件: ['yesterday']


**结果解读**

today 这一项让错误比较可见。全是未来卡片的用例只能拦住漏掉筛选的实现，不能拦住 `<` 与 `<=` 的差别；一组回归应覆盖不同错误类型。

**小练习**

把 limit 设为 1 后还能看出差别吗？怎样安排输入与 limit 让边界被实际检查到？

<details><summary>完成后展开参考分析</summary>

昨天的卡占掉唯一名额，两种实现输出相同。可增大 limit 或只放今天及未来卡，保证边界项进入可观察范围。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 4. 用 CLI 保存，另一个进程继续

每次调用下面的 CLI 都是一个新 Python 进程。执行与读取结果是实际发生的，分层提示来自题目配置；这个离线步骤不冒充新的模型调用。

你也可以把自己写的练习放进 .local/student-submissions，再显式选择文件提交。不要把不信任的下载脚本放进去运行。


In [6]:
directory=ROOT/".local/student-runs/l09/notebook";directory.mkdir(parents=True,exist_ok=True)
command=[sys.executable, '-X', 'utf8',str(PROJECT/"src/feedback_cli.py"),"--db",str(directory/"learning.sqlite3"),"--exercise","due-cards"]
first=json.loads(subprocess.check_output(command+["--submission",str(PROJECT/"exercises/due-cards/student.py"),"--hint-request","help-1"],text=True, encoding='utf-8'))
lesson=first["state"]["lesson_id"]
second=json.loads(subprocess.check_output(command+["--lesson-id",lesson,"--submission",str(PROJECT/"exercises/due-cards/teaching_revision.py")],text=True, encoding='utf-8'))
print(first["state"]["status"],first["hint"])
print(second["state"]["status"],len(second["state"]["attempts"]))
assert second["state"]["lesson_id"]==lesson and len(second["state"]["attempts"])==2


needs_work {'level': 1, 'hint': '看失败用例中的 due 是否恰好等于 today，这张卡应不应该被选中？', 'scope': '只提供当前一级提示，不返回完整参考实现'}
tests_passed 2


## 5. 评估沿用第七节，按层定位失败

运行层看超时、预算、工具和恢复；任务层看实际用例与来源；教学层看提示是否合适、是否一次讲太多；迁移层看换资料后是否仍能复现。一次通过不代表稳定成功率。

本次只验证两个课程练习、四次原始反馈任务和两次收窄工具后的日期任务。起始版本与教学修订示例只是可复核的教学案例，不是学习成效数据，不能据此宣传“提高了学习效果”。


In [7]:
criteria={"execution":"实际执行并保存每条用例", "grounding":"导读结论可回查", "hint":"一次请求一级，不给完整实现", "persistence":"旧失败保留，修改后状态可继续", "transfer":"第二资料目录重跑并核验"}
for key,value in criteria.items():print(key,value)
requests=sum(p["state"]["turns"] for c in report["cases"] for p in c["phases"])
print("本次实际请求数:",requests)


execution 实际执行并保存每条用例
grounding 导读结论可回查
hint 一次请求一级，不给完整实现
persistence 旧失败保留，修改后状态可继续
transfer 第二资料目录重跑并核验
本次实际请求数: 9


## 练习：新增一个有区分力的回归

给 due-cards 再补一个“没有到期卡”的输入，或者给 rank-positive 加一项负分。写明它拦住哪一种错误。保持其他条件相同，比较原始版本与修正版。


In [8]:
new_case={"cards":[{"id":"later","due":"2026-10-01"}],"today":"2026-09-07","limit":3}
print("先预测输出:",[])


先预测输出: []


In [9]:
assert module.solve(new_case["cards"],new_case["today"],new_case["limit"])==[]
print("该用例拦住忽略到期筛选、直接返回前几张卡的实现。它不能单独拦住把 <= 写成 < 的错误。")


该用例拦住忽略到期筛选、直接返回前几张卡的实现。它不能单独拦住把 <= 写成 < 的错误。


## 6. 输出迁移与回归报告

迁移报告要把“复用流程”和“重建契约”分开记录，并用会改变输出的回归用例证明边界真的被检查。失败还要能定位到执行、任务、反馈或迁移层。


In [10]:
out=ROOT/".local/student-runs/l09";out.mkdir(parents=True,exist_ok=True)
transfer_report={
    "artifact":"transfer-and-regression-report",
    "recorded_at":datetime.now(timezone.utc).isoformat(),
    "source_exercise":"rank-positive","target_exercise":"due-cards",
    "reused_process":["显式练习契约","独立进程执行","逐项测试证据","分层提示","状态持久化","回归复核"],
    "rebuilt_contract":["输入字段从 score 改为 due","边界从正分改为 due <= today","排序键与有效日期前提重新定义"],
    "discriminating_case":{"name":"equal-today","correct":["yesterday","today"],"mutant":["yesterday"],
                           "blocked_error":"把 <= today 错写成 < today"},
    "cross_process_continuation":{"lesson_id":lesson,"attempt_count":len(second["state"]["attempts"]),
                                  "status":second["state"]["status"]},
    "failure_layers":{"execution":"进程、超时、权限与恢复","task":"契约、来源与逐项结果",
                      "feedback":"提示级别、事实与答案泄漏","transfer":"目标契约是否重建并通过有区分力的回归"},
    "conclusion_boundary":"案例证明流程可复用与状态可继续，不证明通用辅导效果或学习提升"
}
json_path=out/"transfer-and-regression-report.json";md_path=out/"transfer-and-regression-report.md"
json_path.write_text(json.dumps(transfer_report,ensure_ascii=False,indent=2),encoding="utf-8")
md_path.write_text("\n".join(["# 迁移与回归报告","",f"- 源练习：{transfer_report['source_exercise']}",
    f"- 目标练习：{transfer_report['target_exercise']}","- 复用流程："+"、".join(transfer_report["reused_process"]),
    "- 重建契约："+"、".join(transfer_report["rebuilt_contract"]),
    f"- 有区分力的用例：{transfer_report['discriminating_case']['name']}",
    f"- 跨进程继续：{transfer_report['cross_process_continuation']['status']}",
    f"- 结论边界：{transfer_report['conclusion_boundary']}"]),encoding="utf-8")
print(json_path);print(md_path)


.local\student-runs\l09\transfer-and-regression-report.json
.local\student-runs\l09\transfer-and-regression-report.md


## 本节与综合实践总结

从资料地图到有来源的导读，再到小练习、实际执行、一级提示、修改和持续学习；最后用第二个仓库检验复用范围。完整项目既保存成功也保存失败。

最后一节直播用这些材料答疑、讲评典型问题，并说明考核与提交要求。
